# Module 03 — Storage I/O Performance

Compare read/write performance across formats.

In [ ]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import begin_module
from shared import config
from shared.utils import timed_action, drop_iceberg_table

spark = begin_module('module-03-storage-io')
trips = spark.read.parquet(*config.yellow_taxi_paths())
sample = trips

## Read Benchmarks

In [ ]:
out = config.OUTPUT_BASE / '03-read-bench'
out.mkdir(parents=True, exist_ok=True)

drop_iceberg_table(spark, 'local.default.taxi_sample')

sample.write.mode('overwrite').json(str(out / 'json'))
sample.write.mode('overwrite').parquet(str(out / 'parquet'))
sample.writeTo('local.default.taxi_sample').using('iceberg').create()

In [ ]:
results = {}

_, t = timed_action(spark.read.json(str(out / 'json')))
results['json'] = t

spark.conf.set('spark.sql.parquet.enableVectorizedReader', 'false')
_, t = timed_action(spark.read.parquet(str(out / 'parquet')))
results['parquet_serial'] = t

spark.conf.set('spark.sql.parquet.enableVectorizedReader', 'true')
_, t = timed_action(spark.read.parquet(str(out / 'parquet')))
results['parquet_vectorized'] = t

_, t = timed_action(spark.table('local.default.taxi_sample'))
results['iceberg'] = t

for fmt, sec in sorted(results.items(), key=lambda x: x[1]):
    print(f'{fmt:20s} {sec:.2f}s')

## Write Benchmarks

In [ ]:
write_out = config.OUTPUT_BASE / '03-write-bench'
write_out.mkdir(parents=True, exist_ok=True)
drop_iceberg_table(spark, 'local.default.taxi_write_bench')

write_results = {}
import time

for fmt, writer in [
    ('json', lambda: sample.write.mode('overwrite').json(str(write_out / 'json'))),
    ('csv', lambda: sample.write.mode('overwrite').option('header', True).csv(str(write_out / 'csv'))),
    ('parquet', lambda: sample.write.mode('overwrite').parquet(str(write_out / 'parquet'))),
    ('iceberg', lambda: sample.writeTo('local.default.taxi_write_bench').using('iceberg').create()),
]:
    start = time.perf_counter()
    writer()
    write_results[fmt] = time.perf_counter() - start

for fmt, sec in sorted(write_results.items(), key=lambda x: x[1]):
    print(f'{fmt:10s} write: {sec:.2f}s')

## Key Takeaways

1. Columnar formats outperform row-based.
2. Vectorized Parquet reader is faster than serial.
3. Iceberg adds metadata overhead on write but enables pruning on read.